In [13]:
# Import libraries
import numpy as np
import torch
import torch.nn as nn

In [14]:
# Numpy self attention
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

def self_attention_numpy(X, Wq, Wk, Wv):

    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    dk = Q.shape[-1]

    scores = (Q @ K.T) / np.sqrt(dk)
    weights = softmax(scores)

    output = weights @ V

    return output

In [15]:
# Create inputs
np.random.seed(0)
torch.manual_seed(0)

seq_len = 4
d_model = 8

X = np.random.rand(seq_len, d_model).astype(np.float32)

Wq = np.random.rand(d_model, d_model).astype(np.float32)
Wk = np.random.rand(d_model, d_model).astype(np.float32)
Wv = np.random.rand(d_model, d_model).astype(np.float32)

In [16]:
# Run numpy self attention
numpy_output = self_attention_numpy(X, Wq, Wk, Wv)

In [19]:
# Pytorch multihead attention
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, bias=False, batch_first=True)

Wq_t = torch.tensor(Wq.T, dtype=torch.float32)
Wk_t = torch.tensor(Wk.T, dtype=torch.float32)
Wv_t = torch.tensor(Wv.T, dtype=torch.float32)

mha.in_proj_weight.data = torch.cat([Wq_t, Wk_t, Wv_t], dim=0)

# Disable output projection
mha.out_proj.weight.data = torch.eye(d_model)

X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(0)

torch_output, _ = mha(X_t, X_t, X_t)

torch_output = torch_output.squeeze(0).detach().numpy()

In [20]:
# Compare outputs
print("Difference:")
print(np.abs(numpy_output - torch_output).mean())

Difference:
1.421758165712217e-07


In [ ]:
'''A very small difference (~1e-7) shows that the NumPy implementation correctly reproduces the behavior of PyTorch nn.MultiheadAttention.
This confirms the formula is implemented correctly and that only minor floating-point rounding differences exist.'''